# Cosine top-K

In [ ]:
#|default_exp utils.cosine

In [ ]:
#|export
import asyncio

import numpy as np

from ai_index.utils._model_config import _split_remote_kwargs, _strip_remote_kwargs

def cosine_topk(
    A: np.ndarray,
    B: np.ndarray,
    k: int = 5,
    *,
    mode: str = "local",
    **kwargs,
) -> dict:
    """Compute top-K cosine similarity (api/local/sbatch).

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).

    Returns:
        Dict with "indices" (n, k) and "scores" (n, k) arrays.
    """
    slurm_accounting = kwargs.pop("slurm_accounting", None)

    if mode in ("api", "local"):
        from llm_runner.cosine import run_cosine_topk
        if mode == "api":
            kwargs.setdefault("device", "cpu")
        _strip_remote_kwargs(kwargs)
        return run_cosine_topk(A, B, k, **kwargs)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import run_remote
        cfg = dict(kwargs)
        remote_kw = _split_remote_kwargs(cfg)
        remote_kw.setdefault("job_name", "cosine_topk")
        cfg["k"] = k
        result = run_remote(
            "cosine_topk",
            inputs={"A": A, "B": B},
            config_dict=cfg,
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result

    else:
        raise ValueError(f"Unknown mode: {mode!r}. cosine_topk supports 'api', 'local' and 'sbatch'.")

In [ ]:
#|export
async def acosine_topk(
    A: np.ndarray,
    B: np.ndarray,
    k: int = 5,
    *,
    mode: str = "local",
    **kwargs,
) -> dict:
    """Async version of cosine_topk.

    For api/local mode, runs in a thread. For sbatch, uses arun_remote.

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).
    """
    slurm_accounting = kwargs.pop("slurm_accounting", None)

    if mode in ("api", "local"):
        from llm_runner.cosine import run_cosine_topk
        if mode == "api":
            kwargs.setdefault("device", "cpu")
        _strip_remote_kwargs(kwargs)
        return await asyncio.to_thread(run_cosine_topk, A, B, k, **kwargs)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import arun_remote
        cfg = dict(kwargs)
        remote_kw = _split_remote_kwargs(cfg)
        remote_kw.setdefault("job_name", "cosine_topk")
        cfg["k"] = k
        result = await arun_remote(
            "cosine_topk",
            inputs={"A": A, "B": B},
            config_dict=cfg,
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result

    else:
        raise ValueError(f"Unknown mode: {mode!r}. cosine_topk supports 'api', 'local' and 'sbatch'.")